# Trabalho Final — Análise Avançada de Dados
## Quais fatores mais influenciam o salário de um profissional de dados no Brasil em 2024?

**Centro Universitário Santo Agostinho — UNIFSA**
**Disciplina:** Análise Avançada de Dados — Profa. Ma. Heloísa Guimarães
**Dupla:** João William e Guilherme Frazão

**Base:** *State of Data Brazil 2024–2025* (Data Hackers + Bain & Company) — 5.217 respondentes, 400+ colunas.
Fonte: https://www.kaggle.com/datasets/datahackers/state-of-data-brazil-20242025

---

### Como este notebook está organizado
1. **Carregamento e reconhecimento** — shape, dtypes, nulos e renomeação das colunas-chave.
2. **Limpeza e decisões documentadas** — conversão das faixas salariais, tratamento de *missings* e padronização de categorias.
3. **Análise univariada** — distribuição e estatísticas descritivas de ≥4 variáveis.
4. **Análise bivariada** — ≥3 cruzamentos com o salário + correlações.
5. **Síntese visual** — um gráfico que responde diretamente à pergunta central.

> **Nota metodológica:** cada bloco de limpeza traz uma célula Markdown com o **critério** adotado e a **consequência** da escolha. Uma limpeza sem justificativa não tem valor analítico.


---
### ▶️ Como rodar no Google Colab
1. Abra este `.ipynb` no Colab (Arquivo → Abrir notebook → Upload).
2. Menu **Ambiente de execução → Executar tudo**.
3. Na célula de carregamento, se o download automático não funcionar, o Colab abrirá um botão de **upload** — selecione o CSV baixado do Kaggle.
4. Ao final, baixe o `state_of_data_tratado.csv` gerado (painel de arquivos à esquerda) para alimentar o dashboard.


## Etapa 0 — Configuração do ambiente

Importamos as bibliotecas e definimos um padrão visual único para todos os gráficos (paleta e tamanho), garantindo identidade visual consistente — o mesmo padrão será reaproveitado no dashboard.

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

# Identidade visual única do trabalho
PALETA = ["#1F4E79", "#2E75B6", "#9DC3E6", "#C55A11", "#7F7F7F"]
sns.set_theme(style="whitegrid", palette=PALETA)
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.dpi": 110,
})
print("Ambiente configurado.")

## Etapa 1 — Carregamento e reconhecimento

Carregamos o CSV e inspecionamos *shape*, tipos e nulos. As colunas dessa base vêm com nomes em formato de tupla, como `('P2_h ', 'Faixa salarial')`. Em vez de fixar os nomes exatos, criamos **funções que localizam cada coluna** pelo rótulo, pelo código (`P1_a`, `P2_h`...) ou até pelos valores que ela contém — deixando o notebook robusto a pequenas variações de nomenclatura.

In [ ]:
# === Carregamento do CSV — otimizado para Google Colab (plug-and-play) ===
import os, glob

def carregar_csv():
    """Tenta, em ordem: CSV local -> download via Kaggle -> upload manual."""
    # 1) CSV já presente na pasta (Colab: /content)
    locais = glob.glob("*.csv") + glob.glob("dados/*.csv") + glob.glob("/content/*.csv")
    locais = [c for c in locais if "tratado" not in c.lower()]
    if locais:
        return locais[0]
    # 2) Download automático via kagglehub (requer estar logado no Kaggle)
    try:
        try:
            import kagglehub
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
            import kagglehub
        pasta = kagglehub.dataset_download("datahackers/state-of-data-brazil-20242025")
        csvs = glob.glob(os.path.join(pasta, "**", "*.csv"), recursive=True)
        if csvs:
            return csvs[0]
    except Exception as e:
        print("Download automático indisponível:", e)
    # 3) Upload manual pelo Colab
    try:
        from google.colab import files
        print(">> Faça upload do CSV baixado do Kaggle:")
        enviados = files.upload()
        return list(enviados.keys())[0]
    except Exception:
        raise FileNotFoundError(
            "CSV não encontrado. Baixe em kaggle.com/datasets/datahackers/"
            "state-of-data-brazil-20242025 e coloque na pasta /content (Colab)."
        )

caminho = carregar_csv()
print("Lendo:", caminho)
df = pd.read_csv(caminho, low_memory=False)
print("Shape (linhas, colunas):", df.shape)

In [ ]:
# Achatamento defensivo: se as colunas vierem como tuplas reais, vira string única
df.columns = [str(c) for c in df.columns]
df.head(3)

In [ ]:
# Tipos e amostra de nomes de colunas
print("Tipos de dados (resumo):")
print(df.dtypes.value_counts(), "\n")
print("Exemplos de nomes de coluna:")
for c in list(df.columns)[:8]:
    print("  -", c)

In [ ]:
# Percentual de valores nulos (top 15 colunas com mais nulos)
nulos = (df.isna().mean() * 100).sort_values(ascending=False)
nulos.head(15).round(1).to_frame("% nulos")

### Funções de detecção de colunas

`achar_coluna` procura por trechos de texto no nome da coluna (rótulo ou código).
`achar_por_valores` identifica a coluna pelo **conteúdo** (ex.: a coluna de senioridade é a que contém "Júnior/Pleno/Sênior"). Isso torna a renomeação reprodutível mesmo se o cabeçalho mudar entre edições da pesquisa.

In [ ]:
def achar_coluna(df, *termos):
    """Retorna o nome da 1ª coluna cujo nome contém algum dos termos (case-insensitive)."""
    for termo in termos:
        for col in df.columns:
            if termo.lower() in col.lower():
                return col
    return None

def achar_por_valores(df, valores_alvo, min_match=2):
    """Retorna a 1ª coluna cujos valores únicos contêm >= min_match dos valores_alvo."""
    alvo = {v.lower() for v in valores_alvo}
    for col in df.columns:
        try:
            vals = {str(v).lower().strip() for v in df[col].dropna().unique()}
        except Exception:
            continue
        if len(alvo & vals) >= min_match:
            return col
    return None

### Renomeação de ≥10 colunas-chave

Selecionamos as colunas relevantes para a pergunta central (perfil, cargo, formação e salário) e damos a elas nomes limpos e em português. As demais 380+ colunas permanecem na base, mas não serão usadas — manter o foco é parte da análise.

In [ ]:
mapa = {}

def registrar(nome_novo, coluna_orig):
    if coluna_orig is not None:
        mapa[coluna_orig] = nome_novo

# Cada chamada tenta vários termos (rótulos com espaço E códigos com underscore da base 2024)
registrar("idade",            achar_coluna(df, "1.a_idade", "P1_a '", "'Idade'"))
registrar("genero",           achar_coluna(df, "_genero", "nero", "Genero"))
registrar("cor_raca",         achar_coluna(df, "cor/raca", "etnia", "Cor/raça"))
registrar("uf",               achar_coluna(df, "uf_onde_mora", "uf onde", "estado_onde"))
registrar("regiao",           achar_coluna(df, "regiao_onde_mora", "Regiao onde", "região onde"))
registrar("nivel_ensino",     achar_coluna(df, "nivel_de_ensino", "Nivel de Ensino", "escolaridade"))
registrar("area_formacao",    achar_coluna(df, "área_de_formação", "area_de_forma", "Área de Forma"))
registrar("setor",            achar_coluna(df, "_setor", "Setor", "setor de mercado"))
registrar("cargo",            achar_coluna(df, "cargo_atual", "Cargo atual", "funcao_de_atuacao"))
registrar("tempo_exp_dados",  achar_coluna(df, "experiencia_em_dados", "experiência na área de dados", "tempo_de_experiencia"))
registrar("gestor",           achar_coluna(df, "atua_como_gestor", "Atualmente lidera", "é gestor"))

# Senioridade: tenta código/rótulo; senão detecta pelos valores
col_sen = achar_coluna(df, "2.g_nivel", "P2_g ") or achar_por_valores(df, ["Júnior", "Pleno", "Sênior"])
registrar("nivel_senioridade", col_sen)

# Salário: tenta rótulo (espaço e underscore); senão pelos valores
col_sal = achar_coluna(df, "faixa_salarial", "Faixa salarial") or \
          achar_por_valores(df, ["de R$ 4.001/mês a R$ 6.000/mês", "Acima de R$ 40.001/mês"])
registrar("faixa_salarial", col_sal)

print(f"{len(mapa)} colunas detectadas e renomeadas:\n")
for orig, novo in mapa.items():
    print(f"  {novo:<22} <-  {orig}")

faltando = [n for n in ['idade','genero','cor_raca','nivel_senioridade','faixa_salarial']
            if n not in mapa.values()]
if faltando:
    print("\n[ATENÇÃO] Não localizei automaticamente:", faltando)
    print("Verifique df.columns e ajuste a função registrar().")

In [ ]:
# Aplica a renomeação e cria um DataFrame de trabalho só com as colunas-chave
df = df.rename(columns=mapa)
colunas_chave = [v for v in mapa.values()]
dados = df[colunas_chave].copy()
print("DataFrame de trabalho:", dados.shape)
dados.head()

## Etapa 2 — Limpeza e decisões documentadas

Três tratamentos centrais: (1) converter as faixas salariais em valor numérico; (2) tratar os *missings* — **sem imputar** gênero e raça; (3) padronizar as categorias de senioridade e cargo.

### Decisão 2.1 — Conversão das faixas salariais para ponto médio

**Critério.** A coluna de salário é texto (`"de R$ 4.001/mês a R$ 6.000/mês"`). Para qualquer estatística numérica precisamos de um número. Adotamos o **ponto médio** de cada faixa:
- Faixa fechada `de A a B` → `(A + B) / 2` (ex.: 4.001–6.000 → **5.000**).
- Faixa aberta inferior `Menos de N` → `N × 0,5`.
- Faixa aberta superior `Acima de N` → `N × 1,2` (margem arbitrária, assumida explicitamente).

**Consequência / limitação.** O ponto médio assume distribuição uniforme dentro da faixa, o que **não é verdade** (a distribuição salarial é assimétrica à direita). Isso **subestima** levemente a cauda alta e introduz imprecisão. Por isso reportaremos a **mediana** como medida principal, mais robusta a esse efeito e a *outliers*.

In [ ]:
def faixa_para_ponto_medio(faixa):
    if pd.isna(faixa):
        return np.nan
    s = str(faixa).lower()
    nums = re.findall(r"\d[\d\.]*", s)
    nums = [int(n.replace(".", "")) for n in nums if n.replace(".", "").isdigit()]
    if not nums:
        return np.nan
    aberta_inf = ("menos" in s) or ("até" in s and len(nums) == 1)
    aberta_sup = ("acima" in s) or ("mais de" in s)
    if aberta_inf:
        return nums[0] * 0.5
    if aberta_sup:
        return nums[0] * 1.2
    if len(nums) >= 2:
        return (nums[0] + nums[1]) / 2
    return float(nums[0])

dados["salario_num"] = dados["faixa_salarial"].apply(faixa_para_ponto_medio)

# Conferência: faixa original -> valor numérico
(dados[["faixa_salarial", "salario_num"]]
    .dropna()
    .drop_duplicates()
    .sort_values("salario_num"))

### Decisão 2.2 — Tratamento de valores ausentes

**Critério.**
- **Salário:** removemos as linhas sem faixa salarial — sem a variável-resposta não há análise possível para este trabalho.
- **Gênero e raça/cor:** **não imputamos.** A não-resposta nesses campos é uma escolha do respondente e, em si, uma informação. Imputar pela moda distorceria justamente as análises de desigualdade. Mantemos a categoria `"Não informado"`.

**Consequência.** Excluir linhas sem salário reduz o N e pode introduzir viés se quem não informa salário for sistematicamente diferente (ex.: desempregados). Registramos quantas linhas saem. Para gênero/raça, o viés de não-resposta é discutido na seção de ética do relatório.

In [ ]:
n_antes = len(dados)
dados = dados.dropna(subset=["salario_num"]).copy()
print(f"Linhas sem salário removidas: {n_antes - len(dados)}  ({(n_antes-len(dados))/n_antes:.1%})")
print(f"N final para análise: {len(dados)}")

for c in ["genero", "cor_raca"]:
    if c in dados.columns:
        dados[c] = dados[c].fillna("Não informado")
        print(f"\n{c} — não-resposta: "
              f"{(dados[c]=='Não informado').mean():.1%} marcados como 'Não informado'")

### Decisão 2.3 — Padronização de categorias

**Critério.**
- **Senioridade** vira uma variável **ordinal** ordenada (Júnior < Pleno < Sênior), permitindo correlação de Spearman.
- **Cargo** é normalizado (minúsculas, sem espaços extras) e agrupado por sinônimos via mapeamento explícito, para que "Data Analyst" e "analista de dados jr" caiam no mesmo grupo.

**Consequência.** O agrupamento de cargos envolve julgamento; títulos ambíguos podem ser classificados de forma imperfeita. Por isso o mapeamento fica **explícito e auditável** abaixo.

In [ ]:
# --- Senioridade ordinal ---
ORDEM_SEN = ["Júnior", "Pleno", "Sênior"]
if "nivel_senioridade" in dados.columns:
    dados["nivel_senioridade"] = dados["nivel_senioridade"].astype("category")
    presentes = [n for n in ORDEM_SEN if n in dados["nivel_senioridade"].cat.categories]
    if presentes:
        dados["nivel_senioridade"] = dados["nivel_senioridade"].cat.set_categories(presentes, ordered=True)
    print("Senioridade (ordenada):", list(dados["nivel_senioridade"].cat.categories))
    print(dados["nivel_senioridade"].value_counts(dropna=False))

In [ ]:
# --- Padronização de cargo por palavras-chave ---
def padroniza_cargo(valor):
    if pd.isna(valor):
        return "Não informado"
    t = str(valor).lower()
    regras = [
        ("Engenheiro de Dados",      ["engenheiro de dados", "data engineer", "engenharia de dados"]),
        ("Cientista de Dados",       ["cientista", "data scientist", "ciência de dados"]),
        ("Analista de Dados",        ["analista de dados", "data analyst", "analista de bi"]),
        ("Analista de BI/BI",        ["business intelligence", "analista de bi", "bi "]),
        ("Engenheiro de ML/IA",      ["machine learning", "ml engineer", "engenheiro de ia", "inteligência artificial"]),
        ("Analytics Engineer",       ["analytics engineer"]),
        ("Gestor/Liderança",         ["coordenador", "gerente", "head", "gestor", "diretor", "líder", "lead"]),
        ("Analista de Negócios",     ["analista de negócios", "business analyst", "product"]),
        ("DBA/Infra de Dados",       ["dba", "administrador de banco", "database"]),
    ]
    for nome, chaves in regras:
        if any(k in t for k in chaves):
            return nome
    return "Outros"

if "cargo" in dados.columns:
    dados["cargo_padrao"] = dados["cargo"].apply(padroniza_cargo)
    print(dados["cargo_padrao"].value_counts())

## Etapa 3 — Análise univariada

Para cada variável principal: distribuição de frequências e, no caso numérico (salário), o conjunto completo de estatísticas — **média, mediana, moda, desvio-padrão, variância, amplitude, assimetria e curtose** — com histograma e boxplot.

### 3.1 Salário (variável numérica central)

In [ ]:
s = dados["salario_num"]
desc = {
    "N":            int(s.count()),
    "Média":        s.mean(),
    "Mediana":      s.median(),
    "Moda":         s.mode().iloc[0],
    "Desvio-padrão":s.std(),
    "Variância":    s.var(),
    "Mínimo":       s.min(),
    "Máximo":       s.max(),
    "Amplitude":    s.max() - s.min(),
    "Assimetria":   s.skew(),
    "Curtose":      s.kurtosis(),
}
resumo_sal = pd.Series(desc).round(2)
print(resumo_sal.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(s, bins=30, color=PALETA[1], edgecolor="white")
axes[0].axvline(s.mean(),   color=PALETA[3], ls="--", lw=2, label=f"Média: R$ {s.mean():,.0f}")
axes[0].axvline(s.median(), color=PALETA[0], ls="-",  lw=2, label=f"Mediana: R$ {s.median():,.0f}")
axes[0].set_title("Distribuição do salário mensal (ponto médio das faixas)")
axes[0].set_xlabel("Salário (R$/mês)"); axes[0].set_ylabel("Frequência"); axes[0].legend()

sns.boxplot(x=s, ax=axes[1], color=PALETA[2])
axes[1].set_title("Boxplot do salário mensal")
axes[1].set_xlabel("Salário (R$/mês)")
plt.tight_layout(); plt.show()

**Interpretação.** A assimetria positiva (à direita) confirma a expectativa: a maioria concentra-se em faixas intermediárias e uma minoria de salários muito altos estica a cauda. Por isso **média > mediana**, e a **mediana** é a medida de tendência central mais honesta para reportar. A curtose elevada indica cauda mais pesada que a normal.

### 3.2 a 3.4 — Variáveis categóricas (senioridade, gênero, cargo, região)

In [ ]:
def barras_frequencia(coluna, titulo, ax, ordem=None):
    vc = dados[coluna].value_counts()
    if ordem is not None:
        vc = vc.reindex([o for o in ordem if o in vc.index])
    sns.barplot(x=vc.values, y=vc.index, ax=ax, color=PALETA[1])
    ax.set_title(titulo); ax.set_xlabel("Nº de respondentes"); ax.set_ylabel("")
    for i, v in enumerate(vc.values):
        ax.text(v, i, f" {v}", va="center")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
if "nivel_senioridade" in dados: barras_frequencia("nivel_senioridade", "Senioridade", axes[0,0], ORDEM_SEN)
if "genero" in dados:            barras_frequencia("genero", "Gênero", axes[0,1])
if "cargo_padrao" in dados:      barras_frequencia("cargo_padrao", "Cargo (padronizado)", axes[1,0])
reg = "regiao" if "regiao" in dados else ("uf" if "uf" in dados else None)
if reg:                          barras_frequencia(reg, f"Distribuição por {reg}", axes[1,1])
plt.tight_layout(); plt.show()

**Interpretação.** Observe a composição da amostra: predominância de determinada senioridade, forte desbalanceamento de gênero e concentração geográfica (Sudeste). Esse desbalanceamento **importa** — qualquer média global é puxada pelo grupo majoritário, o que reforça a necessidade da análise bivariada controlada a seguir.

## Etapa 4 — Análise bivariada

Cruzamos o salário com **senioridade**, **gênero** e **tempo de experiência/região**. Onde a variável é ordinal, calculamos a **correlação de Spearman** (apropriada para dados ordinais e distribuição assimétrica).

### 4.1 Salário × Senioridade

In [ ]:
if "nivel_senioridade" in dados:
    fig, ax = plt.subplots(figsize=(9,5))
    sns.boxplot(data=dados, x="nivel_senioridade", y="salario_num", order=ORDEM_SEN, ax=ax)
    ax.set_title("Salário por senioridade"); ax.set_xlabel(""); ax.set_ylabel("Salário (R$/mês)")
    plt.tight_layout(); plt.show()

    med = dados.groupby("nivel_senioridade")["salario_num"].median()
    print("Mediana salarial por senioridade:\n", med.round(0).to_string())

    cod = {n:i for i,n in enumerate(ORDEM_SEN)}
    tmp = dados.dropna(subset=["nivel_senioridade"]).copy()
    tmp["sen_cod"] = tmp["nivel_senioridade"].map(cod)
    rho, p = stats.spearmanr(tmp["sen_cod"], tmp["salario_num"])
    print(f"\nSpearman (senioridade × salário): rho = {rho:.3f}, p = {p:.2e}")

**Interpretação.** A senioridade é, provavelmente, o **maior** discriminador salarial: a mediana sobe de forma consistente de Júnior → Pleno → Sênior, e o Spearman positivo e significativo confirma a relação monotônica. *(Confirme os números após rodar.)*

### 4.2 Salário × Gênero — e o teste decisivo: a diferença persiste **dentro do mesmo nível**?

Conforme orientação do trabalho, não basta perguntar "há diferença?". Controlamos por senioridade para ver se o *gap* sobrevive entre profissionais do mesmo nível.

In [ ]:
if "genero" in dados:
    principais = dados["genero"].value_counts().head(2).index.tolist()
    sub = dados[dados["genero"].isin(principais)]

    print("Mediana salarial por gênero (global):")
    print(sub.groupby("genero")["salario_num"].median().round(0).to_string())

    fig, ax = plt.subplots(figsize=(10,5))
    sns.boxplot(data=sub, x="nivel_senioridade", y="salario_num",
                hue="genero", order=ORDEM_SEN, ax=ax)
    ax.set_title("Salário por gênero, controlado por senioridade")
    ax.set_xlabel(""); ax.set_ylabel("Salário (R$/mês)"); ax.legend(title="Gênero")
    plt.tight_layout(); plt.show()

    print("\nMediana por gênero DENTRO de cada senioridade:")
    print(sub.pivot_table(index="nivel_senioridade", columns="genero",
                          values="salario_num", aggfunc="median").round(0).to_string())

**Interpretação.** Se a diferença a favor de um gênero **persistir mesmo dentro do mesmo nível**, ela não é explicada apenas por composição de senioridade — é um sinal de desigualdade estrutural que deve ser comunicado com responsabilidade (ver seção de ética do relatório). *(Dimensione o gap em R$ e % após rodar.)*

### 4.3 Salário × Tempo de experiência (e/ou Região)

In [ ]:
# Tempo de experiência (ordinal): tenta ordenar pelas faixas textuais
if "tempo_exp_dados" in dados:
    exp = dados.dropna(subset=["tempo_exp_dados"]).copy()
    ordem_exp = sorted(exp["tempo_exp_dados"].unique(),
                       key=lambda x: float(re.search(r"\d+", str(x)).group()) if re.search(r"\d+", str(x)) else -1)
    exp["exp_cod"] = exp["tempo_exp_dados"].map({v:i for i,v in enumerate(ordem_exp)})
    rho, p = stats.spearmanr(exp["exp_cod"], exp["salario_num"])
    print(f"Spearman (experiência × salário): rho = {rho:.3f}, p = {p:.2e}")

    fig, ax = plt.subplots(figsize=(11,5))
    sns.boxplot(data=exp, x="tempo_exp_dados", y="salario_num", order=ordem_exp, ax=ax)
    ax.set_title("Salário por tempo de experiência em dados")
    ax.set_xlabel(""); ax.set_ylabel("Salário (R$/mês)")
    plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
# Salário × Região (comparação entre grupos)
reg = "regiao" if "regiao" in dados else ("uf" if "uf" in dados else None)
if reg:
    med_reg = dados.groupby(reg)["salario_num"].median().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(9,5))
    sns.barplot(x=med_reg.values, y=med_reg.index, ax=ax, color=PALETA[1])
    ax.set_title(f"Mediana salarial por {reg}"); ax.set_xlabel("Mediana (R$/mês)"); ax.set_ylabel("")
    plt.tight_layout(); plt.show()
    print(med_reg.round(0).to_string())

**Interpretação.** Experiência tende a correlacionar fortemente com salário (próximo da senioridade, pois são relacionadas). A região costuma mostrar o Sudeste à frente, mas atenção: parte da diferença regional pode refletir concentração de cargos sêniores e trabalho remoto.

## Etapa 5 — Síntese visual: a resposta à pergunta central

Um único gráfico que ranqueia os fatores pela **diferença de mediana salarial** que cada um produz — uma forma direta de responder *"quais fatores mais influenciam o salário?"*. Quanto maior a barra, maior o impacto daquele fator sobre o salário mediano.

In [ ]:
def amplitude_mediana(coluna, min_n=30):
    """Diferença entre a maior e a menor mediana salarial entre as categorias da coluna."""
    g = dados.groupby(coluna)["salario_num"]
    medianas = g.median()[g.size() >= min_n]
    if len(medianas) < 2:
        return None
    return medianas.max() - medianas.min()

candidatos = {
    "Senioridade":   "nivel_senioridade",
    "Tempo de exp.": "tempo_exp_dados",
    "Cargo":         "cargo_padrao",
    "Gênero":        "genero",
    "Região":        "regiao" if "regiao" in dados else "uf",
    "Cor/raça":      "cor_raca",
    "Nível de ensino":"nivel_ensino",
}
impactos = {}
for nome, col in candidatos.items():
    if col in dados.columns:
        amp = amplitude_mediana(col)
        if amp is not None:
            impactos[nome] = amp

imp = pd.Series(impactos).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
cores = [PALETA[3] if v == imp.max() else PALETA[1] for v in imp.values]
ax.barh(imp.index, imp.values, color=cores)
for i, v in enumerate(imp.values):
    ax.text(v, i, f" R$ {v:,.0f}", va="center", fontsize=10)
ax.set_title("O que mais move o salário?\nDiferença de mediana entre as categorias de cada fator",
             fontsize=13)
ax.set_xlabel("Amplitude da mediana salarial (R$/mês)")
plt.tight_layout(); plt.show()

print("Ranking de impacto sobre o salário mediano:")
print(imp.sort_values(ascending=False).round(0).to_string())

**Interpretação (resposta direta).** O gráfico ordena os fatores pela diferença salarial que produzem. Espera-se que **senioridade e tempo de experiência** liderem, seguidos de **cargo** e **região**, com **gênero e cor/raça** representando diferenças que — embora menores em amplitude bruta — são as mais sensíveis eticamente, pois indicam desigualdade entre profissionais que deveriam ser equivalentes. A resposta à pergunta central deve combinar **magnitude** (o que o gráfico mostra) com **natureza** da diferença (mérito vs. desigualdade estrutural).

## Exportação para o dashboard (Google Looker Studio)

Salvamos **apenas as colunas tratadas** num CSV enxuto. Suba esse arquivo no Google Sheets e conecte o Looker Studio a ele — evita retrabalho e mantém o fluxo organizado.

In [ ]:
colunas_export = [c for c in [
    "salario_num", "faixa_salarial", "nivel_senioridade", "cargo_padrao",
    "genero", "cor_raca", "regiao", "uf", "tempo_exp_dados",
    "nivel_ensino", "idade", "setor"
] if c in dados.columns]

export = dados[colunas_export].copy()
export.to_csv("state_of_data_tratado.csv", index=False, encoding="utf-8-sig")
print("Arquivo salvo: state_of_data_tratado.csv")
print("Colunas exportadas:", colunas_export)
export.head()

---
### Conclusão

Com a base tratada, respondemos à pergunta central combinando estatística descritiva e bivariada. O notebook documenta cada decisão de limpeza e suas limitações, e a síntese visual ordena os fatores por impacto salarial — material que alimenta diretamente o relatório narrativo e o dashboard.

*Próximo passo: rodar este notebook com o CSV do Kaggle, registrar os números reais e levá-los ao relatório e ao README.*